# Gesture-Controlled Photo Booth: Hyperparameter Tuning, `imgsz` (Colab)
**IAT 360 Computer Vision Project, Group 6.** Tuned by **Ben Nichiporik**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amynguyen2424/Computer-Vision-Project-Group6/blob/Ben/Gesture_YOLO_Tuning_imgsz_Colab.ipynb)

This is the Colab version of `Gesture_YOLO_Tuning_imgsz.ipynb`. It reads the dataset straight from the group Google Drive folder **ComputerVisionProject-Group6-Dataset**, so nothing needs to be downloaded to your computer.

My hyperparameter is **`imgsz`**: the size images are resized to (longest side, in pixels) for training and inference. Everything else stays at the baseline values.

| Run | imgsz | Everything else |
|---|---|---|
| Baseline | 640 | YOLOv8s (COCO), SGD lr0 0.01, 50 epochs, patience 15, batch 16, seed 0, deterministic |
| imgsz_480 | 480 | same as baseline |
| imgsz_800 | 800 | same as baseline |
| imgsz_960 | 960 | same as baseline |

### How to run
1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. If **ComputerVisionProject-Group6-Dataset** isn't already in your *My Drive*, open the shared folder in Google Drive and choose **Organize → Add shortcut → My Drive**.
3. **Runtime → Run all**. Allow access to Google Drive when asked.

By default the notebook **scores the four saved models** from the GitHub repo on the latest Drive data, which takes about 10 minutes. Set `RETRAIN = True` below to retrain imgsz 480 / 800 / 960 from scratch on Colab; that takes roughly 1-1.5 hours on a T4. Results are saved to **My Drive / CV-Group6-results / Ben_imgsz** in your own Drive. Nothing is written to the shared dataset folder.

In [ ]:
RETRAIN = False   # True: retrain imgsz 480 / 800 / 960 here (about 1-1.5 h on a T4). False: score the saved models.
SIZES = [480, 800, 960]
DRIVE_FOLDER = "ComputerVisionProject-Group6-Dataset"
DRIVE_FOLDER_ID = "1etLHDUerYnEui6lrK9L7-p1An5c1jLfO"
RESULTS_DIR = "/content/drive/MyDrive/CV-Group6-results/Ben_imgsz"

## 0. Setup
Installs the same Ultralytics version used for the local runs, clones the project code (branch `Ben`) and mounts Google Drive.

In [ ]:
!pip install -q "ultralytics==8.4.202"
import os
if not os.path.exists("/content/repo"):
    !git clone -q -b Ben https://github.com/Amynguyen2424/Computer-Vision-Project-Group6 /content/repo
%cd /content/repo

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

def find_dataset():
    candidates = [Path("/content/drive/MyDrive") / DRIVE_FOLDER,
                  Path("/content/drive/.shortcut-targets-by-id") / DRIVE_FOLDER_ID / DRIVE_FOLDER]
    candidates += sorted(Path("/content/drive/MyDrive").glob(f"*/{DRIVE_FOLDER}"))
    for c in candidates:
        if (c / "images").is_dir() and (c / "labels").is_dir():
            return c
    raise FileNotFoundError(
        f"Can't find '{DRIVE_FOLDER}' in your Drive. Open the shared folder in Google Drive, choose "
        "Organize -> Add shortcut -> My Drive, then run this cell again.")

DATA = find_dataset()
print("dataset folder:", DATA)

## 1. Build the YOLO dataset from Drive
`prepare_dataset.py` checks every image/label pair and copies them into `dataset/` on this Colab machine (temporary; deleted when the session ends). HaGRID images 001-368 per class are split 320 train / 40 val / 8 test; all of our own photos (369-400) go to test. It also fixes Joseph's class order and skips stray files such as Drive's "Copy of …" duplicates. Copying from Drive takes a few minutes.

In [ ]:
!python prepare_dataset.py --images "{DATA}/images" --labels "{DATA}/labels"

In [ ]:
import json, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

from gesture_eval import BASELINE, CLASSES, evaluate, error_analysis

ROOT = Path.cwd()
pd.set_option("display.precision", 3)
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch the runtime to GPU)")

manifest = pd.read_csv(ROOT / "dataset" / "split_manifest.csv")
pd.crosstab([manifest["split"], manifest["source"]], manifest["class"], margins=True)

## 2. Train the three `imgsz` variants (only if `RETRAIN = True`)
Identical to the baseline except `imgsz` (`workers=2` for Colab). Without retraining, the saved models from the repo are used (`models/imgsz_<size>_best.pt`, trained on an RTX 5060 Ti). The baseline is always the saved `models/baseline_best.pt`, so every comparison uses the same reference model.

In [ ]:
def train_info(results_csv):
    h = pd.read_csv(results_csv)
    h.columns = h.columns.str.strip()
    best = h.loc[h["metrics/mAP50-95(B)"].idxmax()]
    return {"epochs_run": int(len(h)), "best_epoch": int(best["epoch"]),
            "train_minutes": round(float(h["time"].iloc[-1]) / 60, 1)}

for s in SIZES:
    out, weights = ROOT / "results" / f"imgsz_{s}", ROOT / "models" / f"imgsz_{s}_best.pt"
    if not RETRAIN:
        print(f"imgsz {s}: using saved {weights.name}")
        continue
    hp = {**BASELINE, "imgsz": s, "workers": 2}
    model = YOLO("yolov8s.pt")
    model.train(**hp, project=str(ROOT / "runs"), name=f"imgsz_{s}", exist_ok=True)
    run = Path(model.trainer.save_dir)
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(run / "weights" / "best.pt", weights)
    for f in ["results.csv", "results.png", "args.yaml"]:
        shutil.copy2(run / f, out / f)
    info = train_info(run / "results.csv")
    (out / "metrics.json").write_text(json.dumps(
        {"model": "yolov8s.pt (COCO-pretrained) fine-tuned", "changed": {"imgsz": s},
         "hyperparameters": hp, **info}, indent=2))
    print(f"imgsz {s}: {info}")
    del model; torch.cuda.empty_cache()

### 2.1 Training curves

In [ ]:
print(pd.DataFrame({s: train_info(ROOT / "results" / f"imgsz_{s}" / "results.csv") for s in SIZES}).T.rename_axis("imgsz"))
fig, axes = plt.subplots(len(SIZES), 1, figsize=(16, 8 * len(SIZES)))
for ax, s in zip(axes, SIZES):
    ax.imshow(plt.imread(ROOT / "results" / f"imgsz_{s}" / "results.png")); ax.set_title(f"imgsz {s}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Evaluate all four models the same way
`evaluate()` gives the Ultralytics metrics on validation, full test and team-only test. `error_analysis()` scores the team photos at the deployment threshold (conf 0.25, IoU 0.5): per class, per box size, and inference speed. Each model is evaluated at the `imgsz` it was trained at. The team-only test set (our own photos) is the headline number.

In [ ]:
RUNS = {640: ("baseline", ROOT / "models" / "baseline_best.pt")}
RUNS.update({s: (f"imgsz_{s}", ROOT / "models" / f"imgsz_{s}_best.pt") for s in SIZES})

rows, per_class, by_size, team_hands = [], {}, {}, {}
for s, (name, weights) in sorted(RUNS.items()):
    out = ROOT / "results" / name
    overall, pc = evaluate(weights, s, out)
    ea = error_analysis(weights, s, out)
    info = train_info(out / "results.csv")
    val, team = overall.iloc[0], overall.iloc[2]
    rows.append({"run": name, "imgsz": s, **info,
                 "val mAP50": val["mAP50"], "val mAP50-95": val["mAP50-95"],
                 "team P": team["precision"], "team R": team["recall"], "team F1": team["F1"],
                 "team mAP50": team["mAP50"], "team mAP50-95": team["mAP50-95"],
                 "team fist mAP50": pc.loc[("test (team only)", "fist"), "mAP50"],
                 "small-hand recall": ea["by_size"]["recall"].iloc[0],
                 "F1 @ conf 0.25": ea["summary"]["F1"],
                 "inference ms (batch 16)": json.loads((out / "metrics.json").read_text())["val_speed_ms_per_image"]["inference"]})
    per_class[s] = pc.loc["test (team only)", "mAP50"]
    by_size[s] = ea["by_size"]["recall"]
    team_hands[s] = ea["hands"]

summary = pd.DataFrame(rows).set_index("run")
summary.round(4).to_csv(ROOT / "results" / "tuning_imgsz_summary.csv")
summary

### 3.1 Change from the baseline (team-only test)

In [ ]:
metric_cols = ["val mAP50", "val mAP50-95", "team P", "team R", "team F1", "team mAP50",
               "team mAP50-95", "team fist mAP50", "small-hand recall", "F1 @ conf 0.25"]
(summary[metric_cols] - summary.loc["baseline", metric_cols]).drop("baseline").round(3)

### 3.2 Comparison plots

In [ ]:
sizes = sorted(RUNS)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))

ax = axes[0]
for col, mk in [("team mAP50", "o"), ("team mAP50-95", "s"), ("val mAP50-95", "^")]:
    ax.plot(sizes, summary.set_index("imgsz").loc[sizes, col], marker=mk, label=col)
ax.set_title("Overall metrics"); ax.legend()

ax = axes[1]
pcd = pd.DataFrame(per_class).loc[CLASSES]
for c in CLASSES:
    ax.plot(sizes, pcd.loc[c, sizes], marker="o", lw=3 if c == "fist" else 1.5, label=c)
ax.set_title("Team-only mAP50 per class"); ax.legend()

ax = axes[2]
bsd = pd.DataFrame(by_size)
for lbl in bsd.index:
    ax.plot(sizes, bsd.loc[lbl, sizes], marker="o", label=str(lbl))
ax.set_title("Team recall by hand size (conf 0.25)"); ax.legend()

ax = axes[3]
ax.plot(sizes, summary.set_index("imgsz").loc[sizes, "inference ms (batch 16)"], marker="o")
ax.set_title("Inference time (ms / image, batch 16)"); ax.set_ylim(bottom=0)

for ax in axes:
    ax.set_xlabel("imgsz"); ax.set_xticks(sizes); ax.grid(alpha=0.3)
axes[0].axvline(640, color="grey", ls=":")
plt.tight_layout()
fig.savefig(ROOT / "results" / "tuning_imgsz_comparison.png", dpi=110, bbox_inches="tight")
plt.show()

### 3.3 How big are our hands at each `imgsz`?
Each team-photo hand's side in pixels after resizing, approximated as √(box area fraction) × imgsz.

In [ ]:
side = pd.DataFrame({s: np.sqrt(team_hands[640]["area_frac"]) * s for s in sizes})
side.describe(percentiles=[0.1, 0.25, 0.5]).loc[["min", "10%", "25%", "50%"]].round(0).rename_axis("hand side (px)")

### 3.4 The hands the baseline missed
For every team-photo hand the baseline got wrong at conf 0.25: what each `imgsz` model predicted for it.

In [ ]:
key = ["file", "gt_index"]
base = team_hands[640]
wrong = base.loc[~base["correct"], key + ["person", "class", "area_frac"]].copy()
wrong["area %"] = (wrong.pop("area_frac") * 100).round(2)
for s in sizes:
    wrong[f"{s}"] = wrong.merge(team_hands[s], on=key, how="left")["pred_class"].values
fixed = {s: int((wrong[f"{s}"] == wrong["class"]).sum()) for s in sizes}
print(f"baseline errors: {len(wrong)} | fixed by: " + ", ".join(f"imgsz {s}: {n}" for s, n in fixed.items() if s != 640))
wrong.sort_values("area %").reset_index(drop=True)

### 3.5 Control: bigger inference size without retraining
Is the effect from *training* at a larger size or just from *running* at a larger size? The baseline weights (trained at 640) are evaluated on the team photos at the other sizes.

In [ ]:
base_model = YOLO(RUNS[640][1])
control = {}
for s in sizes:
    b = base_model.val(data="data_team_test.yaml", split="test", imgsz=s, batch=16, plots=False,
                       project=str(ROOT / "runs"), name=f"control_infer_{s}", exist_ok=True, verbose=False).box
    control[s] = {"baseline weights, team mAP50": b.map50, "baseline weights, team mAP50-95": b.map,
                  "trained at this size, team mAP50": summary.set_index("imgsz").loc[s, "team mAP50"]}
pd.DataFrame(control).T.rename_axis("imgsz").round(3)

## 4. Save the results to your Drive

In [ ]:
out = Path(RESULTS_DIR)
out.mkdir(parents=True, exist_ok=True)
for name in ["baseline"] + [f"imgsz_{s}" for s in SIZES]:
    shutil.copytree(ROOT / "results" / name, out / name, dirs_exist_ok=True)
for f in ["tuning_imgsz_summary.csv", "tuning_imgsz_comparison.png"]:
    shutil.copy2(ROOT / "results" / f, out / f)
if RETRAIN:
    for s in SIZES:
        shutil.copy2(ROOT / "models" / f"imgsz_{s}_best.pt", out / f"imgsz_{s}_best.pt")
print("saved to", out)

## 5. Discussion
> Written from the local run on the earlier export of the dataset (152-image test set, 120 of our photos). This notebook uses the latest Drive data: Amy's 8 palm labels are now included, and Joseph's boxes were redrawn. That gives 160 test images, 128 of them our photos, so the exact numbers above will differ slightly. Check that the conclusions still hold against the tables above.

| Run | imgsz | Team mAP50 | Team mAP50-95 | Fist mAP50 | Small-hand recall | F1 @ 0.25 | Train min | Inference ms (batch 16) |
|---|---|---|---|---|---|---|---|---|
| imgsz_480 | 480 | 0.769 | 0.368 | 0.440 | 7 / 16 | 0.742 | 5.3 | ≈2.8 |
| **Baseline** | **640** | **0.781** | 0.362 | 0.468 | 9 / 16 | **0.777** | 8.4 | ≈4.4 |
| imgsz_800 | 800 | **0.792** | **0.370** | **0.490** | 9 / 16 | 0.765 | 17.0 | ≈6.7 |
| imgsz_960 | 960 | 0.769 | 0.357 | 0.459 | 8 / 16 | 0.719 | 26.0 | ≈9-12 |

(Timings vary by about ±1-2 ms between runs; batch-1 latency is about 10-12 ms for every size because per-call overhead dominates.)

**Result: the hypothesis was mostly not supported.** Changing `imgsz` moves team-only mAP50 by at most ±0.012, well inside the ±0.06 uncertainty of a single model on our 120 photos (bootstrap, see the baseline report). Validation is saturated (0.990-0.995) for every size, so it can't separate the runs either.

- **imgsz 800 is the best variant on paper**: highest team mAP50 (0.792, +0.012), mAP50-95 (0.370), fist mAP50 (0.490, +0.023) and precision (0.856, +0.030). But its recall is lower (0.714 vs 0.751), so F1 at the deployment threshold is slightly *worse* than the baseline (0.765 vs 0.777). Against the baseline it fixed 6 hands and broke 7.
- **Small hands did not improve.** Recall on hands under 1% of the image is 9/16 for both 640 and 800, 8/16 at 960 and 7/16 at 480. Larger `imgsz` did not fix the baseline's main weakness.
- **imgsz 960 is worst for its cost**: it early-stopped at epoch 41 (best epoch 26), took 3× the training time, fixed 8 hands but broke 18 (large-hand recall fell from 0.80 to 0.69), and F1 dropped to 0.719.
- **imgsz 480** is the fastest (≈2.8 ms) and only slightly worse overall (−0.012 mAP50), but it loses the most on small hands (7/16) and fists.

**Why a bigger `imgsz` didn't help small hands.** Resolution was not the real problem. The training set already has plenty of small hands: 386 of the 1,280 HaGRID training hands (30%) cover under 1% of the image, and the median training hand (1.7% of the image) is *smaller* than ours (3.6%). The gap is **scene context**. HaGRID shows one person standing and facing a phone camera, mostly in a plain indoor room. Our missed hands are in crowds, outdoors, at odd angles and against busy backgrounds. A larger input size gives the model more pixels, but it doesn't teach it what those scenes look like. Two further observations:
- **The control in 3.4 fits this.** Running the 640-trained baseline at a larger inference size *lowers* team mAP50 (0.781 → 0.760 at 800 → 0.723 at 960). Our hands are already larger on average than HaGRID's, and upscaling pushes them further from the sizes seen in training.
- **imgsz 960 may be under-trained.** Its drop on large hands may partly come from training: it early-stopped with its best checkpoint at epoch 26, so it effectively trained for half as long as the others.

The more promising fixes target the training data:
- photos with more varied scenes (crowds, outdoors, several people)
- augmentation that varies the scene (`mosaic`, `mixup`, `scale`, colour changes)

**Choice.** For the group comparison my best `imgsz` is **800** (best team mAP50, mAP50-95 and fist mAP50). The gain over the baseline is within noise, though, and it costs about 2× the training time and 1.5× the inference time. For the photo booth itself I would keep **640**: it has the best F1 at the deployment threshold and the best speed/accuracy balance. If another member's hyperparameter improves scale robustness (e.g. `scale` or `mosaic`), combining it with imgsz 800 is the combination worth testing.
